# Lesson 5: Planning

For each cell: **guess what it will print first**, then press ▶ and check.

From [Build an AI Agent](https://gradlabs.in/learn/agents/lessons/lesson5.html).

In [ ]:
import re

In [ ]:
# Tools: a pretend shop, and the calculator
def shop(item):
    prices = {"cake for 10 at cake house": "₹1800", "snacks for 10": "₹1500"}
    if "sweet spot" in item.lower():
        return "Error: Sweet Spot is closed today"
    return prices.get(item.lower(), f"Error: no price found for {item}")

def calculator(math):
    for ch in math:
        if ch not in "0123456789+-*/(). ":
            return "Error: the calculator only takes numbers and + - * / ( )"
    if "**" in math:   # ** is power: 9 ** 999999 would never finish
        return "Error: the calculator only takes numbers and + - * / ( )"
    try:
        return str(eval(math))
    except Exception:
        return "Error: that math doesn't work out"

TOOLS = {"shop": shop, "calculator": calculator}

INSTRUCTIONS = """You are an agent. TOOLS you can use:
shop(item) - the price of an item at a shop. Give it the item and the shop.
calculator(math) - works out math exactly.
First write PLAN: and your steps, one per line, numbered.
Then do one step at a time: ACT: tool(input)
If a step fails, write a new PLAN:.
When you're done, write: ANSWER: your answer"""

In [ ]:
# Read the numbered steps out of a plan
def read_plan(reply):
    steps = []
    for line in reply.split("\n"):
        if line != "" and line[0] in "0123456789":   # not empty, starts with a number
            steps.append(line)
    return steps

In [ ]:
# A fake LLM with a plan, a failed step and a new plan
fake_replies = [
    "PLAN:\n1. Find a cake for 10 at Sweet Spot\n2. Find snacks for 10\n3. Add up the cost",
    "ACT: shop(cake for 10 at Sweet Spot)",
    "THINK: Step 1 failed. A new plan.\nPLAN:\n1. Find a cake for 10 at Cake House\n2. Find snacks for 10\n3. Add up the cost",
    "ACT: shop(cake for 10 at Cake House)",
    "ACT: shop(snacks for 10)",
    "ACT: calculator(1800 + 1500)",
    "ANSWER: A Cake House cake and snacks: ₹3300 in all, under ₹5000.",
]
def fake_llm(text):
    return fake_replies.pop(0)

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The loop, now keeping track of the plan
def run_agent(question, llm, step_limit=8):
    text = INSTRUCTIONS + "\n\nQuestion: " + question + "\n"
    plan = []
    done = 0
    for step in range(step_limit):
        reply = llm(text)
        print(reply)
        text += reply + "\n"
        if "PLAN:" in reply:
            plan = read_plan(reply)
            done = 0
            print(f"(a plan with {len(plan)} steps)")
        if "ANSWER:" in reply:
            return reply.split("ANSWER:")[1].strip()
        act = re.search(r"ACT: (\w+)\((.*)\)", reply)
        if act is None:
            continue
        elif act.group(1) not in TOOLS:
            look = f"Error: no tool called {act.group(1)}"
        else:
            look = TOOLS[act.group(1)](act.group(2).strip('"'))
        if not look.startswith("Error"):
            done += 1
        print("LOOK:", look, f"(done {done} of {len(plan)})")
        text += "LOOK: " + look + "\n"
    return "Stopped: the step limit was reached."

print(run_agent("Order a cake and snacks for 10 people, under ₹5000.", fake_llm))

## Now a real LLM

The same code, with a small free model (Qwen2.5 1.5B) instead of the fake LLM.
The first run downloads it (about 3 GB), so give it a few minutes.

Small models make mistakes. Watch the `LOOK:` lines: errors go back to the LLM, so it can often fix them.

In [ ]:
# A real LLM: a small, free model that runs inside Colab
# (faster with a GPU: Runtime → Change runtime type → T4 GPU)
import torch
from transformers import pipeline
chat = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
                device=0 if torch.cuda.is_available() else -1)

def real_llm(text):
    out = chat([{"role": "user", "content": text}], max_new_tokens=120, do_sample=False)
    reply = out[0]["generated_text"][-1]["content"]
    kept = []
    for line in reply.split("\n"):
        kept.append(line)
        if line.startswith("ACT:") or line.startswith("ANSWER:"):
            break   # cut here: the program writes the LOOK: line, not the LLM
    return "\n".join(kept)

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The same planning loop, now with a real LLM
print(run_agent("Order a cake and snacks for 10 people, under ₹5000.", real_llm))